#### Initial Data Analysis
- Live EDA on dataset
    - Initial notes: 32 total cols, 7 cols have no data variation, 6 cols missing 79% of data
    - Column headers do not maintain one format, some use snakecase, others user underscores
    - Metadata can be filled in by utilizing the userID
    - Difference b/t timestamp and sessionStart?
        - Timestamp can be split into date/time and binned post-discovery
    - More users are needed for variation
    - What are these columns:
        - sessionDelta (59% data missing)
        - plaid_score (always -100)
        - imei_score (always 90.909090...)
        - blacklist_score (always 100)
        - duplicate_score (always 100)
    - Data errors:
        - Margate, FL & Pompano, FL coords take you to SanFran, CA
        - Miami, FL coords take you to Margate, FL

In [1]:
# Notebook level pip upgrade

%pip install --quiet --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Notebook level library installs

%pip install -q pandas numpy matplotlib plotly seaborn rich jupytext 


Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas
import numpy
import matplotlib.pyplot as plt
import seaborn
from pathlib import Path
import rich
from rich.console import Console
from rich.table import Table

console = Console()

# Pathlib setup for data files
data = Path("C:/Users/theda/OneDrive/Career Development/Kaylus/clip_this/data")
kaylus_raw = data / "01_raw" / "kaylus_case_study.csv"

# Initial library import and data load
console.print("[green]Libraries installed and/or imported: pandas, numpy, matplotlib, seaborn, rich, pathlib[/green]")

Libraries installed and/or imported: pandas, numpy, matplotlib, seaborn, rich, pathlib

In [4]:
# Initialize EDA log for tracking steps and status
pina_eda_log = []

def show_eda_summary(log_data):
    table = Table(title="EDA Tracker", show_header=True, header_style="bold magenta")
    table.add_column("Step", style="dim", width=6)
    table.add_column("Action")
    table.add_column("Status", justify="right")

    for entry in log_data:
        table.add_row(entry['step'], entry['action'], entry['status'])

    console.print(table)

# 2. Helper to add steps easily
'''def add_step(action, status="[green]Complete[/green]"):
    step_num = str(len(pina_eda_log) + 1)
    pina_eda_log.append({"step": step_num, "action": action, "status": status})
    show_eda_summary(pina_eda_log)'''

# Step 2: Updated to handle duplicate steps and allow for status updates
def add_step(action, status="[green]Complete[/green]"):
    # Check if this action is already in the log
    for entry in pina_eda_log:
        if entry["action"] == action:
            entry["status"] = status  # Update the status
            show_eda_summary(pina_eda_log)
            return

    # If it's a new action, append it
    step_num = str(len(pina_eda_log) + 1)
    pina_eda_log.append({"step": step_num, "action": action, "status": status})
    show_eda_summary(pina_eda_log)
# Step 3: Helper to pop the last step if needed    
def pop_step():
    if pina_eda_log:
        pina_eda_log.pop()  # Removes the last item in the list
    show_eda_summary(pina_eda_log)

In [5]:
console.print(f"[white]Dataset file path: {kaylus_raw}[/white]")
df = pandas.read_csv(kaylus_raw)

#First step added to log

add_step("Initialized EDA log and summary table")

Dataset file path: C:\Users\theda\OneDrive\Career Development\Kaylus\clip_this\data\01_raw\kaylus_case_study.csv

                         EDA Tracker                         
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Step   ┃ Action                                ┃   Status ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ 1      │ Initialized EDA log and summary table │ Complete │
└────────┴───────────────────────────────────────┴──────────┘

In [6]:
add_step("Loaded dataset into DataFrame")



                         EDA Tracker                         
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Step   ┃ Action                                ┃   Status ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ 1      │ Initialized EDA log and summary table │ Complete │
│ 2      │ Loaded dataset into DataFrame         │ Complete │
└────────┴───────────────────────────────────────┴──────────┘

In [7]:
df.info()

add_step("Early data inspection with df.info() to check data types and # of missing values")

<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   document_id               125 non-null    str    
 1   timestamp                 125 non-null    str    
 2   userID                    125 non-null    str    
 3   inquisitorScore           125 non-null    float64
 4   sessionStart              92 non-null     str    
 5   sessionDelta              51 non-null     float64
 6   city                      92 non-null     str    
 7   region                    92 non-null     str    
 8   country                   92 non-null     str    
 9   gps_lat                   92 non-null     float64
 10  gps_lng                   92 non-null     float64
 11  plaid_score               125 non-null    int64  
 12  imei_score                125 non-null    float64
 13  blacklist_score           125 non-null    int64  
 14  duplicate_score      

                                              EDA Tracker                                               
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Step   ┃ Action                                                                           ┃   Status ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ 1      │ Initialized EDA log and summary table                                            │ Complete │
│ 2      │ Loaded dataset into DataFrame                                                    │ Complete │
│ 3      │ Early data inspection with df.info() to check data types and # of missing values │ Complete │
└────────┴──────────────────────────────────────────────────────────────────────────────────┴──────────┘

In [8]:
# Some headers are camel, some are snake, some combine different formatting
console.print(f"[white]Current Columns:[/white] {df.columns.tolist()}")
add_step("Listed current columns to understand dataset structure and header formatting")

Current Columns: ['document_id', 'timestamp', 'userID', 'inquisitorScore', 'sessionStart', 'sessionDelta', 'city', 
'region', 'country', 'gps_lat', 'gps_lng', 'plaid_score', 'imei_score', 'blacklist_score', 'duplicate_score', 
'manual_refresh_score', 'geo_checkin_score', 'gesture_complexity_score', 'offer_redeem_score', 
'dailyClippedOffers', 'dailyPotentialSavings', 'sessionClips', 'gesture_timestamp', 'gesture_vx', 'gesture_vy', 
'gesture_combinedVelocity', 'gestureState_moveX', 'gestureState_moveY', 'gestureState_dx', 'gestureState_dy', 
'gestureState_vx', 'gestureState_vy']

                                              EDA Tracker                                               
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Step   ┃ Action                                                                           ┃   Status ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ 1      │ Initialized EDA log and summary table                                            │ Complete │
│ 2      │ Loaded dataset into DataFrame                                                    │ Complete │
│ 3      │ Early data inspection with df.info() to check data types and # of missing values │ Complete │
│ 4      │ Listed current columns to understand dataset structure and header formatting     │ Complete │
└────────┴──────────────────────────────────────────────────────────────────────────────────┴──────────┘

In [ ]:
# Perform the snake_case conversion + check
df.columns = (df.columns
              .str.replace(r'([a-z0-9])([A-Z])', r'\1_\2', regex=True)
              .str.replace(r'([A-Z])([A-Z][a-z])', r'\1_\2', regex=True) # UserID -> User_ID
              .str.strip()
              .str.lower()
              .str.replace(' ', '_', regex=False)
              .str.replace('-', '_', regex=False))


console.print(f"[white]Current Columns:[/white] {df.columns.tolist()}")
add_step("Column Name Standardization to snake_case", "[green]Snake Case Applied[/green]")



Current Columns: ['document_id', 'timestamp', 'user_id', 'inquisitor_score', 'session_start', 'session_delta', 
'city', 'region', 'country', 'gps_lat', 'gps_lng', 'plaid_score', 'imei_score', 'blacklist_score', 
'duplicate_score', 'manual_refresh_score', 'geo_checkin_score', 'gesture_complexity_score', 'offer_redeem_score', 
'daily_clipped_offers', 'daily_potential_savings', 'session_clips', 'gesture_timestamp', 'gesture_vx', 
'gesture_vy', 'gesture_combined_velocity', 'gesture_state_move_x', 'gesture_state_move_y', 'gesture_state_dx', 
'gesture_state_dy', 'gesture_state_vx', 'gesture_state_vy']

                                                   EDA Tracker                                                    
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Step   ┃ Action                                                                           ┃             Status ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ 1      │ Initialized EDA log and summary table                                            │           Complete │
│ 2      │ Loaded dataset into DataFrame                                                    │           Complete │
│ 3      │ Early data inspection with df.info() to check data types and # of missing values │           Complete │
│ 4      │ Listed current columns to understand dataset structure and header formatting     │           Complete │
│ 5      │ Column Name Standardization to snake_case                                        │ Snake Case Applied │
└────────┴──────────────────────────────────────────────────────────────────────────────────┴────────────────────┘